# Calendar Dimension Table

Builds the standard calendar dimension `adb_db_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including date keys, year/quarter/month/week attributes, and period boundary flags.

In [0]:
from pyspark.sql import functions as F

START_DATE = "1940-01-01"
END_DATE = "2030-12-31"
TARGET_TABLE = "adb_db_caresync.silver.calendar"

# One row per day in the range
dates_df = spark.sql(
    f"SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) AS date"
)

d = F.col("date")

calendar_df = (
    dates_df.select(
        F.date_format(d, "yyyyMMdd").cast("int").alias("date_key"),
        d.alias("date"),
        # Year
        F.year(d).alias("year"),
        F.when((F.year(d) % 4 == 0) & ((F.year(d) % 100 != 0) | (F.year(d) % 400 == 0)), True).otherwise(False).alias("is_leap_year"),
        F.trunc(d, "year").alias("first_day_of_year"),
        F.make_date(F.year(d), F.lit(12), F.lit(31)).alias("last_day_of_year"),
        # Quarter
        F.quarter(d).alias("quarter"),
        F.concat(F.lit("Q"), F.quarter(d)).alias("quarter_name"),
        F.concat(F.year(d), F.lit("-Q"), F.quarter(d)).alias("year_quarter"),
        F.trunc(d, "quarter").alias("first_day_of_quarter"),
        F.date_sub(F.add_months(F.trunc(d, "quarter"), 3), 1).alias("last_day_of_quarter"),
        F.datediff(d, F.trunc(d, "quarter")).alias("_dq"),
        # Month
        F.month(d).alias("month"),
        F.date_format(d, "MMMM").alias("month_name"),
        F.date_format(d, "MMM").alias("month_short_name"),
        F.date_format(d, "yyyy-MM").alias("year_month"),
        (F.year(d) * 100 + F.month(d)).alias("year_month_key"),
        F.trunc(d, "month").alias("first_day_of_month"),
        F.last_day(d).alias("last_day_of_month"),
        F.dayofmonth(F.last_day(d)).alias("days_in_month"),
        # Week (ISO 8601: weeks start Monday)
        F.weekofyear(d).alias("week_of_year"),
        F.expr("extract(YEAROFWEEK FROM date)").alias("iso_year"),
        F.ceil(F.dayofmonth(d) / 7).cast("int").alias("week_of_month"),
        F.date_trunc("week", d).cast("date").alias("week_start_date"),
        F.date_add(F.date_trunc("week", d).cast("date"), 6).alias("week_end_date"),
        # Day
        F.dayofmonth(d).alias("day_of_month"),
        F.dayofyear(d).alias("day_of_year"),
        F.expr("extract(DAYOFWEEK_ISO FROM date)").alias("day_of_week"),  # 1=Mon ... 7=Sun
        F.date_format(d, "EEEE").alias("day_name"),
        F.date_format(d, "EEE").alias("day_short_name"),
        # Flags
        F.dayofweek(d).isin(1, 7).alias("is_weekend"),
        (~F.dayofweek(d).isin(1, 7)).alias("is_weekday"),
        (F.dayofmonth(d) == 1).alias("is_month_start"),
        (d == F.last_day(d)).alias("is_month_end"),
        (d == F.trunc(d, "quarter")).alias("is_quarter_start"),
        (d == F.date_sub(F.add_months(F.trunc(d, "quarter"), 3), 1)).alias("is_quarter_end"),
        ((F.month(d) == 1) & (F.dayofmonth(d) == 1)).alias("is_year_start"),
        ((F.month(d) == 12) & (F.dayofmonth(d) == 31)).alias("is_year_end"),
    )
    .withColumn("day_of_quarter", F.col("_dq") + 1)
    .drop("_dq")
)

(
    calendar_df.orderBy("date")
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Wrote {spark.table(TARGET_TABLE).count()} rows to {TARGET_TABLE}")

In [0]:
%sql
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT date_key) AS distinct_keys,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  DATEDIFF(MAX(date), MIN(date)) + 1 AS expected_rows,
  SUM(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM adb_db_caresync.silver.calendar

In [0]:
%sql
SELECT * FROM adb_db_caresync.silver.calendar
WHERE date BETWEEN '2023-12-28' AND '2024-01-03'
ORDER BY date